# Investigating Distributions in NDT7 and Cloudflare AIM Data

This code investigates and compares the distributions of data from NDT7 and Cloudflare AIM. The data is from September 2025. It plots PDFs and computes the JSD between different measurements.


In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
from matplotlib import colormaps

In [2]:
def get_download_and_upload_df(df):
    df_download = df[~df["download_throughput_mbps"].isna()]
    df_upload = df[~df["upload_throughput_mbps"].isna()]
    return df_download, df_upload

ndt7 = pd.read_csv("data/ndt7_sept.csv").dropna(subset=['client_country_code'])

ndt7_download, ndt7_upload = get_download_and_upload_df(ndt7)
cf_data_mean = pd.read_csv("data/cf_mean_sept.csv").dropna(subset=['client_country_code'])
cf_data_median = pd.read_csv("data/cf_median_sept.csv").dropna(subset=['client_country_code'])
cf_data_90th_percentile = pd.read_csv("data/cf_p90_sept.csv").dropna(subset=['client_country_code'])

countries = np.union1d(
    ndt7_download['client_country_code'].unique(),
    cf_data_mean['client_country_code'].unique()
)

download_study_fields = ["packet_loss_rate", "download_throughput_mbps", "download_latency_ms", "download_jitter_ms"]
upload_study_fields = ["packet_loss_rate", "upload_throughput_mbps", "upload_latency_ms", "upload_jitter_ms"]

download_fields_info = {
    "packet_loss_rate": {
        "title": "Packet Loss Rate",
    },
    "download_throughput_mbps": {
        "title": "Download Throughput",
        "measure_unit": "Mbps"
    },
    "download_latency_ms": {
        "title": "Download Latency",
        "measure_unit": "ms"
    },
    "download_jitter_ms": {
        "title": "Download Jitter",
        "measure_unit": "ms"
    }
}

upload_fields_info = {
    "packet_loss_rate": {
        "title": "Packet Loss Rate",
    },
    "upload_throughput_mbps": {
        "title": "Upload Throughput",
        "measure_unit": "Mbps"
    },
    "upload_latency_ms": {
        "title": "Upload Latency",
        "measure_unit": "ms"
    },
    "upload_jitter_ms": {
        "title": "Upload Jitter",
        "measure_unit": "ms"
    }
}

color_for_country = dict(zip(countries, [colormaps['tab20'](i / max(len(countries) - 1, 1)) for i in range(len(countries))]))

In [3]:
def remove_countries_with_few_measurements(ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, threshold=50):
    datasets = {
        "ndt7_download": ndt7_download,
        "ndt7_upload": ndt7_upload,
        "cf_data_mean": cf_data_mean,
        "cf_data_median": cf_data_median,
        "cf_data_90th_percentile": cf_data_90th_percentile,
    }

    countries_to_drop = set()

    print(f"Checking countries with < {threshold} rows:\n")
    for name, df in datasets.items():
        counts = df['client_country_code'].value_counts()
        few_rows = counts[counts < threshold]
        if not few_rows.empty:
            print(f"In {name}:")
            for country, count in few_rows.items():
                print(f"  {country}: {count} rows")
            countries_to_drop.update(few_rows.index)

    countries_array = countries
    if countries_to_drop:
        print(f"\nDropping countries (fewer than 10 rows in any dataset): {sorted(countries_to_drop)}\n")
        for name in datasets:
            datasets[name] = datasets[name][~datasets[name]['client_country_code'].isin(countries_to_drop)]
        countries_array = np.array([c for c in countries if c not in countries_to_drop])
    else:
        print("\nNo countries to drop.\n")

    return countries_array, datasets["ndt7_download"], datasets["ndt7_upload"], datasets["cf_data_mean"], datasets["cf_data_median"], datasets["cf_data_90th_percentile"]

In [4]:
countries, ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile = remove_countries_with_few_measurements(ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, threshold=50)
countries = ['US', 'DE', 'AU']

Checking countries with < 50 rows:

In ndt7_download:
  SL: 47 rows
  MT: 46 rows
  SO: 44 rows
  BT: 41 rows
  MH: 39 rows
  BL: 35 rows
  IL: 32 rows
  NI: 27 rows
  ME: 19 rows
  TT: 18 rows
  BB: 17 rows
  FM: 16 rows
  WS: 14 rows
  BA: 14 rows
  RW: 13 rows
  SI: 13 rows
  IS: 12 rows
  SB: 12 rows
  SR: 11 rows
  MD: 11 rows
  CK: 11 rows
  JO: 10 rows
  TO: 9 rows
  LS: 7 rows
  MP: 6 rows
  TL: 5 rows
  GE: 5 rows
  TV: 5 rows
  AZ: 4 rows
  AG: 4 rows
  VC: 3 rows
  IN: 3 rows
  AS: 2 rows
  BH: 2 rows
  AQ: 2 rows
  DM: 2 rows
  GD: 1 rows
  MV: 1 rows
  SJ: 1 rows
  AM: 1 rows
In ndt7_upload:
  GF: 47 rows
  CI: 43 rows
  SL: 40 rows
  BI: 39 rows
  EE: 38 rows
  VU: 37 rows
  MT: 37 rows
  OM: 33 rows
  MH: 32 rows
  KI: 28 rows
  IL: 27 rows
  BT: 26 rows
  BL: 25 rows
  NI: 25 rows
  BB: 14 rows
  TT: 14 rows
  SO: 13 rows
  SB: 12 rows
  RW: 10 rows
  FM: 10 rows
  IS: 9 rows
  BA: 9 rows
  GE: 9 rows
  MD: 9 rows
  CK: 8 rows
  SR: 8 rows
  SI: 7 rows
  ME: 6 rows
  TO

In [5]:
def get_histogram(data, bin_width=None):
    num_bins = 100
    if bin_width is not None:
        num_bins = int((max(data) - min(data)) / bin_width)
    return np.histogram(data,bins=num_bins, density=True)

In [6]:
def compute_jsd(ndt7_df, cf_mean, cf_median, cf_90th, countries, field):
    results = []
    cf_dfs = {
        "mean": cf_mean,
        "median": cf_median,
        "90th_percentile": cf_90th
    }

    for country in countries:
        ndt_data = ndt7_df[ndt7_df['client_country_code'] == country][[field]].dropna()
        counts_ndt, bin_edges_ndt = get_histogram(ndt_data[field])
        bin_width = bin_edges_ndt[1] - bin_edges_ndt[0]

        for label, cf_df in cf_dfs.items():
            cf_data = cf_df[cf_df['client_country_code'] == country][[field]].dropna()
            if cf_data.empty:
                continue

            counts_cf, _ = get_histogram(cf_data[field], bin_width=bin_width)

            max_len = max(len(counts_ndt), len(counts_cf))
            p = np.zeros(max_len)
            q = np.zeros(max_len)
            p[:len(counts_ndt)] = counts_ndt
            q[:len(counts_cf)] = counts_cf

            js = jensenshannon(p, q, base=np.e) ** 2

            results.append({
                "country": country,
                "cf_source": label,
                "JS Divergence": round(js, 4),
            })

    return pd.DataFrame(results)

In [7]:
def compute_jsd_with_median(ndt7_download, ndt7_upload, cf_median, countries, fields):
    results = []

    for country in countries:
        for field in fields:
            if 'download' in field:
                ndt7_df = ndt7_download
            elif 'upload' in field:
                ndt7_df = ndt7_upload
            else:
                ndt7_df = pd.concat([ndt7_download, ndt7_upload], ignore_index=True)
            ndt_data = ndt7_df[ndt7_df['client_country_code'] == country][[field]].dropna()
            counts_ndt, bin_edges_ndt = get_histogram(ndt_data[field])
            bin_width = bin_edges_ndt[1] - bin_edges_ndt[0]

            cf_data = cf_median[cf_median['client_country_code'] == country][[field]].dropna()

            counts_cf, _ = get_histogram(cf_data[field], bin_width=bin_width)

            max_len = max(len(counts_ndt), len(counts_cf))
            p = np.zeros(max_len)
            q = np.zeros(max_len)
            p[:len(counts_ndt)] = counts_ndt
            q[:len(counts_cf)] = counts_cf

            js = jensenshannon(p, q, base=np.e) ** 2

            results.append({
                "field": field,
                "country": country,
                "JS Divergence": round(js, 4),
            })

    return pd.DataFrame(results)

In [8]:
jsd_countries = ['US', 'BR', 'GB', 'JP']

In [9]:
compute_jsd(ndt7_download, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'download_latency_ms')

,country,cf_source,JS Divergence
0,US,mean,0.0163
1,US,median,0.0426
2,US,90th_percentile,0.0122
3,BR,mean,0.0726
4,BR,median,0.1000
5,BR,90th_percentile,0.0167
6,GB,mean,0.0113
7,GB,median,0.0258
8,GB,90th_percentile,0.0252
9,JP,mean,0.0977


In [10]:
compute_jsd(ndt7_download, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'download_throughput_mbps')


,country,cf_source,JS Divergence
0,US,mean,0.3475
1,US,median,0.1609
2,US,90th_percentile,0.0646
3,BR,mean,0.3716
4,BR,median,0.1469
5,BR,90th_percentile,0.2011
6,GB,mean,0.4088
7,GB,median,0.1794
8,GB,90th_percentile,0.0335
9,JP,mean,0.3897


In [11]:
compute_jsd(ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'upload_latency_ms')


,country,cf_source,JS Divergence
0,US,mean,0.0267
1,US,median,0.0232
2,US,90th_percentile,0.0330
3,BR,mean,0.0690
4,BR,median,0.0768
5,BR,90th_percentile,0.0408
6,GB,mean,0.0123
7,GB,median,0.0128
8,GB,90th_percentile,0.0198
9,JP,mean,0.0299


In [12]:
compute_jsd(ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'upload_throughput_mbps')


,country,cf_source,JS Divergence
0,US,mean,0.0586
1,US,median,0.0358
2,US,90th_percentile,0.1218
3,BR,mean,0.1439
4,BR,median,0.1462
5,BR,90th_percentile,0.2157
6,GB,mean,0.0214
7,GB,median,0.0802
8,GB,90th_percentile,0.0641
9,JP,mean,0.1768


In [13]:
compute_jsd_with_median(ndt7_download, ndt7_upload, cf_data_median, jsd_countries, ['download_jitter_ms', 'upload_jitter_ms', 'packet_loss_rate'])

,field,country,JS Divergence
0,download_jitter_ms,US,0.0401
1,upload_jitter_ms,US,0.0228
2,packet_loss_rate,US,0.0253
3,download_jitter_ms,BR,0.0494
4,upload_jitter_ms,BR,0.0266
5,packet_loss_rate,BR,0.0698
6,download_jitter_ms,GB,0.0669
7,upload_jitter_ms,GB,0.0080
8,packet_loss_rate,GB,0.0387
9,download_jitter_ms,JP,0.0661
